In [32]:
import pandas as pd
import numpy as np
data = pd.read_csv("student-por.csv", sep=",")
print("Dataset Shape:", data.shape)
print(data.head())
data["Pass"] = (data["G3"] >= 10).astype(int)

Dataset Shape: (649, 33)
  school sex  age address famsize Pstatus  Medu  Fedu     Mjob      Fjob  ...  \
0     GP   F   18       U     GT3       A     4     4  at_home   teacher  ...   
1     GP   F   17       U     GT3       T     1     1  at_home     other  ...   
2     GP   F   15       U     LE3       T     1     1  at_home     other  ...   
3     GP   F   15       U     GT3       T     4     2   health  services  ...   
4     GP   F   16       U     GT3       T     3     3    other     other  ...   

  famrel freetime  goout  Dalc  Walc health absences  G1  G2  G3  
0      4        3      4     1     1      3        4   0  11  11  
1      5        3      3     1     1      3        2   9  11  11  
2      4        3      2     2     3      3        6  12  13  12  
3      3        2      2     1     1      5        0  14  14  14  
4      4        3      2     1     2      5        0  11  13  13  

[5 rows x 33 columns]


In [33]:
# SELECT FEATURES
X_linear = data[["studytime"]].values
features = [
    "age",
    "studytime",
    "failures",
    "famrel",
    "freetime",
    "goout",
    "Dalc",
    "Walc",
    "health",
    "absences",
    "G1",
    "G2"
]
X = data[features].values
y = data["Pass"].values

In [34]:
# TRAIN-TEST SPLIT MANUALLY
np.random.seed(42)
n = len(X)
indices = np.arange(n)
np.random.shuffle(indices)
train_size = int(0.8 * n)
train_indices = indices[:train_size]
test_indices = indices[train_size:]
X_linear_train = X_linear[train_indices]
X_linear_test = X_linear[test_indices]
y_train = y[train_indices]
y_test = y[test_indices]
X_train = X[train_indices]
X_test = X[test_indices]

In [35]:
# STANDARDIZE FEATURES
mean = np.mean(X_train, axis=0)

std = np.std(X_train, axis=0)
std[std == 0] = 1

X_train_scaled = (X_train - mean) / std

X_test_scaled = (X_test - mean) / std


In [36]:
# ADD BIAS COLUMN
ones_train = np.ones((X_train_scaled.shape[0], 1))
ones_test = np.ones((X_test_scaled.shape[0], 1))
X_train_scaled = np.hstack((ones_train, X_train_scaled))
X_test_scaled = np.hstack((ones_test, X_test_scaled))

In [37]:
# LINEAR REGRESSION
x_mean = np.mean(X_linear_train)
y_mean = np.mean(y_train)
numerator = np.sum(
    (X_linear_train[:, 0] - x_mean) *
    (y_train - y_mean)
)
denominator = np.sum(
    (X_linear_train[:, 0] - x_mean) ** 2
)
b1 = numerator / denominator
b0 = y_mean - (b1 * x_mean)
print("LINEAR REGRESSION")
print("Intercept:", b0)
print("Coefficient:", b1)
linear_predictions = (
    b0 + b1 * X_linear_test[:, 0]
)
linear_classes = (linear_predictions >= 0.5).astype(int)

LINEAR REGRESSION
Intercept: 0.689454349475907
Coefficient: 0.07709609652547876


In [38]:
# MULTIPLE LINEAR REGRESSION
XTX = np.dot(
    X_train_scaled.T,
    X_train_scaled
)
XTX_inverse = np.linalg.pinv(XTX)
XTy = np.dot(
    X_train_scaled.T,
    y_train
)
beta_multiple = np.dot(
    XTX_inverse,
    XTy
)
print("MULTIPLE LINEAR REGRESSION")
print("Coefficients:")
print(beta_multiple)
multiple_predictions = np.dot(
    X_test_scaled,
    beta_multiple
)
multiple_classes = (
    multiple_predictions >= 0.5
).astype(int)

MULTIPLE LINEAR REGRESSION
Coefficients:
[ 0.83815029  0.00099648 -0.00140488 -0.05888661 -0.01958255 -0.00695426
 -0.00391064  0.00712366 -0.0154929   0.02393561  0.00155954  0.05542465
  0.15457422]


In [39]:
# RIDGE REGRESSION
lambda_ridge = 1.0

identity = np.eye(
    X_train_scaled.shape[1]
)
identity[0, 0] = 0

ridge_matrix = (
    XTX +
    lambda_ridge * identity
)
ridge_inverse = np.linalg.pinv(
    ridge_matrix
)
beta_ridge = np.dot(
    ridge_inverse,
    XTy
)
print("RIDGE REGRESSION")
print("Lambda:", lambda_ridge)
print("Coefficients:")
print(beta_ridge)
ridge_predictions = np.dot(
    X_test_scaled,
    beta_ridge
)
ridge_classes = (
    ridge_predictions >= 0.5
).astype(int)

RIDGE REGRESSION
Lambda: 1.0
Coefficients:
[ 0.83815029  0.00100273 -0.00135094 -0.05884493 -0.0194853  -0.00696629
 -0.00390909  0.0070389  -0.01541911  0.02383502  0.00153625  0.05601971
  0.15374503]


In [40]:
# LASSO REGRESSION
def soft_threshold(value, threshold):

    if value > threshold:
        return value - threshold

    elif value < -threshold:
        return value + threshold

    else:
        return 0.0

def lasso_regression(
    X,
    y,
    lambda_value,
    iterations
):

    rows = X.shape[0]
    columns = X.shape[1]
    beta = np.zeros(columns)
    for iteration in range(iterations):
        for j in range(columns):
            # Prediction using all current coefficients
            prediction = np.dot(
                X,
                beta
            )
            # Residual excluding current feature
            residual = (
                y -
                prediction +
                X[:, j] * beta[j]
            )
            numerator = np.dot(
                X[:, j],
                residual
            )
            denominator = np.dot(
                X[:, j],
                X[:, j]
            )
            if j == 0:
                # Intercept is not regularized
                beta[j] = numerator / denominator
            else:
                value = (
                    numerator /
                    denominator
                )
                threshold = (
                    lambda_value /
                    (2 * denominator)
                )
                beta[j] = soft_threshold(
                    value,
                    threshold
                )
    return beta

lambda_lasso = 0.1
beta_lasso = lasso_regression(
    X_train_scaled,
    y_train,
    lambda_lasso,
    1000
)
print("LASSO REGRESSION")
print("Lambda:", lambda_lasso)
print("Coefficients:")
print(beta_lasso)
lasso_predictions = np.dot(
    X_test_scaled,
    beta_lasso
)
lasso_classes = (
    lasso_predictions >= 0.5
).astype(int)

LASSO REGRESSION
Lambda: 0.1
Coefficients:
[ 0.83815029  0.00084311 -0.00123357 -0.05875081 -0.01947073 -0.00690032
 -0.00386512  0.00685002 -0.01518963  0.02379439  0.00145381  0.05534263
  0.15451698]


In [41]:
# 11. CONFUSION MATRIX
def confusion_matrix(
    actual,
    predicted
):
    TP = 0
    TN = 0
    FP = 0
    FN = 0
    for i in range(len(actual)):

        if actual[i] == 1 and predicted[i] == 1:
            TP += 1

        elif actual[i] == 0 and predicted[i] == 0:
            TN += 1

        elif actual[i] == 0 and predicted[i] == 1:
            FP += 1

        elif actual[i] == 1 and predicted[i] == 0:
            FN += 1

    return TP, TN, FP, FN

In [42]:
# CLASSIFICATION METRICS
def calculate_metrics(
    actual,
    predicted
):
    TP, TN, FP, FN = confusion_matrix(
        actual,
        predicted
    )

    total = TP + TN + FP + FN

    accuracy = (
        (TP + TN) /
        total
    )
    if TP + FP == 0:
        precision = 0
    else:
        precision = (
            TP /
            (TP + FP)
        )

    if TP + FN == 0:
        recall = 0
    else:
        recall = (
            TP /
            (TP + FN)
        )

    if precision + recall == 0:
        f1 = 0
    else:
        f1 = (
            2 *
            precision *
            recall /
            (precision + recall)
        )

    return (
        TP,
        TN,
        FP,
        FN,
        accuracy,
        precision,
        recall,
        f1
    )

In [43]:
# CALCULATE ALL RESULT
linear_result = calculate_metrics(
    y_test,
    linear_classes
)
multiple_result = calculate_metrics(
    y_test,
    multiple_classes
)
ridge_result = calculate_metrics(
    y_test,
    ridge_classes
)
lasso_result = calculate_metrics(
    y_test,
    lasso_classes
)

In [44]:
# DISPLAY RESULTS
def print_result(
    name,
    result
):
    TP, TN, FP, FN, accuracy, precision, recall, f1 = result
    print(name)
    print("Confusion Matrix")
    print("TP:", TP)
    print("TN:", TN)
    print("FP:", FP)
    print("FN:", FN)
    print("\nAccuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)
print_result(
    "Linear Regression",
    linear_result
)
print_result(
    "Multiple Linear Regression",
    multiple_result
)
print_result(
    "Ridge Regression",
    ridge_result
)
print_result(
    "LASSO Regression",
    lasso_result
)

Linear Regression
Confusion Matrix
TP: 114
TN: 0
FP: 16
FN: 0

Accuracy : 0.8769230769230769
Precision: 0.8769230769230769
Recall   : 1.0
F1 Score : 0.9344262295081968
Multiple Linear Regression
Confusion Matrix
TP: 113
TN: 4
FP: 12
FN: 1

Accuracy : 0.9
Precision: 0.904
Recall   : 0.9912280701754386
F1 Score : 0.9456066945606694
Ridge Regression
Confusion Matrix
TP: 113
TN: 4
FP: 12
FN: 1

Accuracy : 0.9
Precision: 0.904
Recall   : 0.9912280701754386
F1 Score : 0.9456066945606694
LASSO Regression
Confusion Matrix
TP: 113
TN: 4
FP: 12
FN: 1

Accuracy : 0.9
Precision: 0.904
Recall   : 0.9912280701754386
F1 Score : 0.9456066945606694


In [45]:
# COMPARISON TABLE
comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Multiple Linear Regression",
        "Ridge Regression",
        "LASSO Regression"
    ],
    "Accuracy": [
        linear_result[4],
        multiple_result[4],
        ridge_result[4],
        lasso_result[4]
    ],
    "Precision": [
        linear_result[5],
        multiple_result[5],
        ridge_result[5],
        lasso_result[5]
    ],
    "Recall": [
        linear_result[6],
        multiple_result[6],
        ridge_result[6],
        lasso_result[6]
    ],
    "F1 Score": [
        linear_result[7],
        multiple_result[7],
        ridge_result[7],
        lasso_result[7]
    ]
})
print("FINAL COMPARISON")
print(comparison)

FINAL COMPARISON
                        Model  Accuracy  Precision    Recall  F1 Score
0           Linear Regression  0.876923   0.876923  1.000000  0.934426
1  Multiple Linear Regression  0.900000   0.904000  0.991228  0.945607
2            Ridge Regression  0.900000   0.904000  0.991228  0.945607
3            LASSO Regression  0.900000   0.904000  0.991228  0.945607
